# L39 -- Cats vs Dogs Classification with PyTorch
## A Comparative Study of 10 CNN Architectures

This notebook trains **10 different CNN models** on the Cats vs Dogs dataset and compares their performance. It is fully self-contained and designed to run on **Google Colab with GPU**.

**Research Groups:**
- **Group A (Depth Study):** Baseline, Shallow, Deep, Wide
- **Group B (Regularization):** Dropout, BatchNorm
- **Group C (Architecture):** Small FC, LeNet-Style
- **Group D (Advanced):** Transfer ResNet18, Lightweight

## Section 1: Setup & Configuration

### 1.1 Install Dependencies

First we install all the Python packages we need. Think of packages as toolboxes -- each one has special tools for a specific job. For example, `torch` is our main deep-learning toolbox, and `matplotlib` helps us draw graphs.

In [1]:
# Install all required packages (most are pre-installed on Colab)
!pip install -q torch torchvision numpy matplotlib seaborn scikit-learn tqdm Pillow

### 1.2 Import Libraries

Now we open all our toolboxes and take out the tools we need. Each `import` line is like saying "I want to use this tool in my project."

In [2]:
import os
import time
import random
import zipfile
import shutil
import urllib.request
from pathlib import Path
from typing import Optional

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch import Tensor
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision import models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# Show plots inline in the notebook
%matplotlib inline

print("All libraries imported successfully!")

All libraries imported successfully!


### 1.3 GPU Detection & Seed Setting

A GPU (Graphics Processing Unit) is like a super-fast calculator that can do thousands of math problems at once. Training on GPU is 10-50x faster than CPU.

Setting a "seed" is like picking a starting point for random numbers. If everyone uses the same seed, they get the same random numbers -- making experiments **reproducible**.

In [3]:
# =============================================
# CONSTANTS -- All "magic numbers" in one place
# =============================================
IMAGE_SIZE: int = 150          # Resize all images to 150x150 pixels
NUM_CLASSES: int = 2           # Two categories: cat and dog
CLASS_NAMES: list = ['cat', 'dog']
BATCH_SIZE: int = 32           # Process 32 images at a time
EPOCHS: int = 15               # Go through the entire dataset 15 times
LR: float = 0.001              # Learning rate -- how big each learning step is
SEED: int = 42                 # Random seed for reproducibility
EARLY_STOPPING_PATIENCE: int = 5  # Stop if no improvement for 5 epochs
GRAPH_DPI: int = 150           # Resolution for saved plots

# Color scheme -- one color per model for consistent graphs
MODEL_COLORS: list = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
]

# Research group colors
GROUP_COLORS: dict = {
    "A": "#1f77b4",  # Blue  -- Depth study
    "B": "#2ca02c",  # Green -- Regularization
    "C": "#ff7f0e",  # Orange -- Architecture
    "D": "#d62728",  # Red   -- Advanced
}

# Model metadata
MODEL_NAMES: dict = {
    1: "Baseline CNN", 2: "Shallow CNN", 3: "Deep CNN", 4: "Wide CNN",
    5: "Small FC CNN", 6: "Dropout CNN", 7: "BatchNorm CNN",
    8: "LeNet-Style CNN", 9: "Transfer ResNet18", 10: "Lightweight CNN",
}

MODEL_GROUPS: dict = {
    1: "A", 2: "A", 3: "A", 4: "A",
    5: "C", 6: "B", 7: "B", 8: "C",
    9: "D", 10: "D",
}

# --- GPU Detection ---
def get_device() -> torch.device:
    """Pick the best available device -- GPU if possible, otherwise CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()

# --- Set all random seeds for reproducibility ---
def set_seeds(seed: int = SEED) -> None:
    """Lock down all sources of randomness so results are repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds()

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found! Training will be slow on CPU.")
print(f"PyTorch version: {torch.__version__}")
print(f"Random seed: {SEED}")

### 1.4 Create Output Directories

We need folders to save our graphs and results. This is like creating filing cabinets before we start working.

In [ ]:
# Create output directories for saving plots
RESULTS_DIR = Path("results")
GRAPHS_DIR = RESULTS_DIR / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Graphs will be saved to: {GRAPHS_DIR.resolve()}")

## Section 2: Dataset -- Download & Prepare the Images

### 2.1 Download the Dataset

We are using the **Kaggle Cats and Dogs** dataset (hosted by Microsoft). This Colab notebook uses a large subset: **~20,000 training images** (10,000 cats + 10,000 dogs) and **~5,000 validation images** (2,500 cats + 2,500 dogs) for a total of **~25,000 images**.

Think of it like a school test: training images are the textbook (the model studies them), and validation images are the exam (the model has never seen them before).

> **Note:** The local version of this project uses only ~3,000 images for CPU training speed.

In [ ]:
# Microsoft mirror of the full Kaggle Cats and Dogs dataset
KAGGLE_URL = (
    "https://download.microsoft.com/download/3/E/1/"
    "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
)

def download_dataset(data_dir: Path = Path("data")) -> Path:
    """
    Download the Kaggle Cats & Dogs dataset and organise it into
    train/ and validation/ splits.

    For Colab (GPU): uses up to 10,000 images per class for training.
    Returns the path to the dataset root with train/ and validation/.
    """
    data_dir.mkdir(parents=True, exist_ok=True)
    dataset_root = data_dir / "cats_and_dogs_filtered"

    # Skip if already downloaded and organised
    if (dataset_root / "train" / "cats").exists():
        print(f"Dataset already exists at {dataset_root}")
        return dataset_root

    zip_path = data_dir / "kagglecatsanddogs.zip"

    if not zip_path.exists():
        print("Downloading dataset from Microsoft mirror (~800MB)...")
        req = urllib.request.Request(KAGGLE_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(str(zip_path), "wb") as f:
            shutil.copyfileobj(resp, f)
        print("Download complete!")

    # Extract
    print("Extracting dataset...")
    with zipfile.ZipFile(str(zip_path), "r") as zf:
        zf.extractall(str(data_dir))

    zip_path.unlink()

    # Reorganise: PetImages/Cat|Dog -> train/cats|dogs + validation/cats|dogs
    source = data_dir / "PetImages"
    # Colab gets more data: 10000 train + 2500 val per class
    for split, count in [("train", 10000), ("validation", 2500)]:
        for class_name, src_folder in [("cats", "Cat"), ("dogs", "Dog")]:
            dest_dir = dataset_root / split / class_name
            dest_dir.mkdir(parents=True, exist_ok=True)
            src_dir = source / src_folder
            copied = 0
            for img_path in sorted(src_dir.iterdir()):
                if copied >= count:
                    break
                if not img_path.suffix.lower() in (".jpg", ".jpeg", ".png"):
                    continue
                try:
                    img = Image.open(img_path)
                    img.verify()
                except Exception:
                    continue
                shutil.copy2(str(img_path), str(dest_dir / f"{class_name[:-1]}.{copied}.jpg"))
                copied += 1
            print(f"  {split}/{class_name}: {copied} images")

    # Clean up raw data
    shutil.rmtree(str(source), ignore_errors=True)
    # Also clean up any MACOSX or extra folders
    macosx = data_dir / "__MACOSX"
    if macosx.exists():
        shutil.rmtree(str(macosx), ignore_errors=True)

    print(f"\nDataset ready at {dataset_root}")
    return dataset_root

# Download now!
dataset_root = download_dataset()
print(f"\nTrain cats: {len(list((dataset_root / 'train' / 'cats').glob('*.jpg')))} images")
print(f"Train dogs: {len(list((dataset_root / 'train' / 'dogs').glob('*.jpg')))} images")
print(f"Val cats:   {len(list((dataset_root / 'validation' / 'cats').glob('*.jpg')))} images")
print(f"Val dogs:   {len(list((dataset_root / 'validation' / 'dogs').glob('*.jpg')))} images")

### 2.2 Custom Dataset Class

PyTorch needs a special `Dataset` class that knows how to load one image at a time. Think of it like a librarian -- you ask for image #42, and it finds the file, opens it, resizes it, and hands it back as numbers the model can understand.

In [ ]:
class CatsDogsDataset(Dataset):
    """
    Custom PyTorch Dataset for cat and dog images.
    Each image is loaded, transformed, and returned as a
    (image_tensor, label) pair.  label: 0 = cat, 1 = dog.
    """

    def __init__(
        self,
        root_dir: Path,
        transform: Optional[T.Compose] = None,
        max_samples: Optional[int] = None,
    ) -> None:
        """
        Args:
            root_dir:    folder containing 'cats/' and 'dogs/' subfolders
            transform:   image transforms to apply
            max_samples: limit how many images to load (None = use all)
        """
        self.transform = transform
        self.samples: list = []  # list of (file_path, label_index) tuples

        # Walk through each class folder and collect image paths
        for label_idx, class_name in enumerate(CLASS_NAMES):
            class_dir = root_dir / f"{class_name}s"  # 'cats' or 'dogs'
            if not class_dir.exists():
                print(f"WARNING: Missing directory {class_dir}")
                continue
            files = sorted(class_dir.glob("*.jpg"))
            for fpath in files:
                self.samples.append((fpath, label_idx))

        # Limit samples if requested (take equal from each class)
        if max_samples is not None and len(self.samples) > max_samples:
            cats = [(p, l) for p, l in self.samples if l == 0][:max_samples // 2]
            dogs = [(p, l) for p, l in self.samples if l == 1][:max_samples // 2]
            self.samples = cats + dogs

        print(f"  Loaded {len(self.samples)} images from {root_dir}")

    def __len__(self) -> int:
        """Return total number of images."""
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple:
        """Load one image, apply transforms, return (tensor, label)."""
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except (OSError, IOError) as e:
            print(f"WARNING: Corrupted image {img_path}: {e}")
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            image = self.transform(image)
        return image, label

### 2.3 Image Transforms (Data Augmentation)

**Training transforms** include random flips and rotations -- like showing the model photos from different angles. This helps it learn to recognize cats and dogs no matter how they are positioned.

**Validation transforms** only resize the image -- no tricks -- so the test is fair.

In [ ]:
def get_train_transforms() -> T.Compose:
    """Build the transform pipeline for training images with augmentation."""
    return T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),       # Make all images the same size
        T.RandomHorizontalFlip(p=0.5),            # Flip left-right 50% of the time
        T.RandomRotation(degrees=15),             # Tilt up to 15 degrees
        T.ColorJitter(brightness=0.2, contrast=0.2),  # Slightly change colors
        T.ToTensor(),                              # Convert to numbers (0-1)
        T.Normalize(mean=[0.5, 0.5, 0.5],        # Center values around zero
                    std=[0.5, 0.5, 0.5]),
    ])


def get_val_transforms() -> T.Compose:
    """Build the transform pipeline for validation images (no augmentation)."""
    return T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5],
                    std=[0.5, 0.5, 0.5]),
    ])

print("Transforms defined!")
print("Train: Resize -> RandomFlip -> RandomRotation -> ColorJitter -> ToTensor -> Normalize")
print("Val:   Resize -> ToTensor -> Normalize")

### 2.4 Create DataLoaders

A DataLoader is like a conveyor belt in a factory. Instead of feeding images one at a time, it groups them into **batches** (groups of 32) and feeds them to the model efficiently.

In [ ]:
# Create datasets -- use ALL available images (no max_samples limit for Colab)
train_dataset = CatsDogsDataset(
    root_dir=dataset_root / "train",
    transform=get_train_transforms(),
    max_samples=None,  # Use all images
)
val_dataset = CatsDogsDataset(
    root_dir=dataset_root / "validation",
    transform=get_val_transforms(),
    max_samples=None,  # Use all images
)

# Colab uses 2 workers for faster data loading
num_workers = 2 if torch.cuda.is_available() else 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,       # Randomize order each epoch
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,      # Keep order fixed for evaluation
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False,
)

print(f"\nTrain: {len(train_dataset)} images in {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} images in {len(val_loader)} batches")
print(f"Batch size: {BATCH_SIZE}")

### 2.5 Visualize Sample Images

Let us look at some actual images from our dataset. This is always a good first step -- never train a model without looking at the data first!

In [ ]:
def tensor_to_image(tensor: Tensor) -> np.ndarray:
    """Convert a normalized image tensor back to a displayable NumPy array."""
    img = tensor.cpu().numpy()
    # Un-normalize from [-1, 1] back to [0, 1]
    img = img * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    if img.shape[0] == 1:
        return img.squeeze(0)  # Grayscale
    return np.transpose(img, (1, 2, 0))  # RGB: (C, H, W) -> (H, W, C)


def plot_sample_images(dataloader: DataLoader, num_images: int = 16) -> None:
    """Show a 4x4 grid of sample images with their labels."""
    images, labels = next(iter(dataloader))
    images = images[:num_images]
    labels = labels[:num_images]

    cols = 4
    rows = num_images // cols
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i in range(num_images):
        img = tensor_to_image(images[i])
        axes[i].imshow(img)
        axes[i].set_title(CLASS_NAMES[labels[i].item()], fontsize=12)
        axes[i].axis("off")

    plt.suptitle("Sample Images from Training Set", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / "sample_images.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()

plot_sample_images(train_loader)

### 2.6 Class Distribution

A balanced dataset (roughly 50/50 split between cats and dogs) is important. If we had 90% cats, the model could just always guess "cat" and be 90% right without learning anything!

In [ ]:
def plot_class_distribution(dataset: CatsDogsDataset) -> None:
    """Bar chart showing how many cats vs dogs are in the dataset."""
    labels = [label for _, label in dataset.samples]
    counts = [labels.count(0), labels.count(1)]

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(CLASS_NAMES, counts, color=["#ff7f0e", "#1f77b4"])
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                str(count), ha="center", fontweight="bold")

    ax.set_title("Class Distribution (Training Set)", fontsize=14, fontweight="bold")
    ax.set_ylabel("Number of Images")
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / "class_distribution.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()

plot_class_distribution(train_dataset)

## Section 3: All 10 Model Definitions

We define 10 different CNN architectures to compare. Think of each model as a different "recipe" for recognizing cats and dogs -- some are simple, some are complex, and we want to find out which recipe works best.

| # | Model | Key Idea | Group |
|---|-------|----------|-------|
| 1 | Baseline CNN | Standard 4-layer CNN | A |
| 2 | Shallow CNN | Only 2 layers -- how little can we get away with? | A |
| 3 | Deep CNN | 6 layers -- does more depth help? | A |
| 4 | Wide CNN | Fewer but wider layers | A |
| 5 | Small FC CNN | Tiny classifier head | C |
| 6 | Dropout CNN | Baseline + Dropout regularization | B |
| 7 | BatchNorm CNN | Baseline + Batch Normalization | B |
| 8 | LeNet-Style | Classic 1998 architecture | C |
| 9 | Transfer ResNet18 | Pretrained on ImageNet | D |
| 10 | Lightweight CNN | Global Average Pooling, fewest params | D |

### Model 1: Baseline CNN

This is our **starting point** -- a straightforward 4-layer CNN. Every other model is compared against this baseline.

**Architecture:** 4 convolutional layers that progressively extract features (edges -> shapes -> patterns -> objects), followed by fully-connected layers that make the final cat-or-dog decision.

```
Input (3, 150, 150) -> Conv(32) -> Pool -> Conv(64) -> Pool -> Conv(128) -> Pool -> Conv(128) -> Pool -> Flatten -> FC(512) -> FC(2)
```
~3.45M parameters

In [ ]:
class BaselineCNN(nn.Module):
    """Model 1: Baseline 4-layer CNN -- our starting point for comparison."""

    def __init__(self, input_channels: int = 3) -> None:
        super().__init__()
        # Feature extractor: 4 conv layers with ReLU activation and MaxPool
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3),   # 150->148
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 148->74
            nn.Conv2d(32, 64, kernel_size=3),                # 74->72
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 72->36
            nn.Conv2d(64, 128, kernel_size=3),               # 36->34
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 34->17
            nn.Conv2d(128, 128, kernel_size=3),              # 17->15
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 15->7
        )
        # Classifier: flatten the feature maps + two fully-connected layers
        self.classifier = nn.Sequential(
            nn.Flatten(),                    # (batch, 128, 7, 7) -> (batch, 6272)
            nn.Linear(128 * 7 * 7, 512),     # 6272 -> 512
            nn.ReLU(),
            nn.Linear(512, NUM_CLASSES),      # 512 -> 2
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        return self.classifier(x)

### Model 2: Shallow CNN

The simplest possible CNN -- only **2 convolutional layers**. This tests the question: "How little depth can we get away with?"

With padding=1, the conv layers preserve spatial dimensions before pooling. The tradeoff: the feature map stays large, leading to a huge first FC layer.

~10.6M parameters (most in the FC layer due to large feature maps)

In [ ]:
class ShallowCNN(nn.Module):
    """Model 2: Minimal 2-layer CNN -- tests the floor of depth."""

    def __init__(self, input_channels: int = 3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),  # 150->150
            nn.ReLU(),
            nn.MaxPool2d(2),                                           # 150->75
            nn.Conv2d(32, 64, kernel_size=3, padding=1),              # 75->75
            nn.ReLU(),
            nn.MaxPool2d(2),                                           # 75->37
        )
        # After two pool layers with padding: 150/2/2 = 37 (rounds down from 75/2)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 37 * 37, 128),    # Large input because feature map is big
            nn.ReLU(),
            nn.Linear(128, NUM_CLASSES),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        return self.classifier(x)

### Model 3: Deep CNN

Goes **deeper** than the baseline with **6 convolutional layers**. More layers means the model can learn more complex patterns -- but also risks "vanishing gradients" (the learning signal fading away).

Layers are grouped in pairs, with pooling after each pair to reduce spatial dimensions.

~4-5M parameters

In [ ]:
class DeepCNN(nn.Module):
    """Model 3: 6-layer deep CNN -- tests the impact of extra depth."""

    def __init__(self, input_channels: int = 3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: two conv layers then pool
            nn.Conv2d(input_channels, 32, kernel_size=3),  # 150->148
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3),              # 148->146, pool->73
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3),             # 73->71
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3),            # 71->69, pool->34
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3),            # 34->32
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3),            # 32->30, pool->15
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # After all pooling: (256, 15, 15)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 15 * 15, 512),
            nn.ReLU(),
            nn.Linear(512, NUM_CLASSES),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        return self.classifier(x)

### Model 4: Wide CNN

Instead of going deep, this model goes **wide** -- each layer has more filters (64, 128, 256) to capture more patterns at each level. Tests: "Is it better to have more filters or more layers?"

~38M parameters (wider layers = many more weights)

In [ ]:
class WideCNN(nn.Module):
    """Model 4: 3-layer wide CNN -- more filters per layer than baseline."""

    def __init__(self, input_channels: int = 3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=3),    # 150->148, 64 filters
            nn.ReLU(),
            nn.MaxPool2d(2),                                  # 148->74
            nn.Conv2d(64, 128, kernel_size=3),               # 74->72
            nn.ReLU(),
            nn.MaxPool2d(2),                                  # 72->36
            nn.Conv2d(128, 256, kernel_size=3),              # 36->34
            nn.ReLU(),
            nn.MaxPool2d(2),                                  # 34->17
        )
        # After 3 conv+pool: (256, 17, 17)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 17 * 17, 512),
            nn.ReLU(),
            nn.Linear(512, NUM_CLASSES),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        return self.classifier(x)

### Model 5: Small FC CNN

Same convolutional backbone as a 3-layer CNN, but the "brain" at the end (fully connected layers) is much **smaller**: 128 neurons instead of 512. Tests if we really need a big classifier.

Think of it like having great eyes but a small brain -- can it still make good decisions?

~1.7M parameters

In [ ]:
class SmallFCCNN(nn.Module):
    """Model 5: CNN with a tiny fully-connected head."""

    def __init__(self, input_channels: int = 3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3),    # 150->148
            nn.ReLU(),
            nn.MaxPool2d(2),                                  # 148->74
            nn.Conv2d(32, 64, kernel_size=3),                # 74->72
            nn.ReLU(),
            nn.MaxPool2d(2),                                  # 72->36
            nn.Conv2d(64, 128, kernel_size=3),               # 36->34
            nn.ReLU(),
            nn.MaxPool2d(2),                                  # 34->17
        )
        # Small head: 128 neurons instead of 512
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 17 * 17, 128),   # Much smaller than usual
            nn.ReLU(),
            nn.Linear(128, NUM_CLASSES),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        return self.classifier(x)

### Model 6: Dropout CNN

Same architecture as the baseline, but with **Dropout** layers added. Dropout randomly "turns off" some neurons during training -- like studying with a blindfold sometimes. This forces the model to not rely too much on any single neuron, helping it generalize better.

- `Dropout(0.5)` = 50% of neurons turned off after flatten
- `Dropout(0.3)` = 30% turned off after the first FC layer

~3.45M parameters (same as baseline -- dropout adds no new weights)

In [ ]:
class DropoutCNN(nn.Module):
    """Model 6: Baseline CNN + Dropout -- tests regularization."""

    def __init__(self, input_channels: int = 3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 128, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),                  # Turn off 50% of neurons randomly
            nn.Linear(128 * 7 * 7, 512),
            nn.ReLU(),
            nn.Dropout(0.3),                  # Turn off 30% of neurons randomly
            nn.Linear(512, NUM_CLASSES),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        return self.classifier(x)

### Model 7: BatchNorm CNN

Baseline architecture with **Batch Normalization** after every conv layer. BatchNorm is like a "reset button" -- after each convolution it adjusts all the numbers so they stay in a nice range. This helps the model train faster and more stably.

Order per block: Conv -> BatchNorm -> ReLU -> MaxPool

~3.46M parameters (slightly more due to BN parameters)

In [ ]:
class BatchNormCNN(nn.Module):
    """Model 7: Baseline CNN + BatchNorm after every conv layer."""

    def __init__(self, input_channels: int = 3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3),
            nn.BatchNorm2d(32),          # Normalize outputs after conv
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 128, kernel_size=3),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, NUM_CLASSES),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        return self.classifier(x)

### Model 8: LeNet-Style CNN

Inspired by Yann LeCun's **LeNet-5** (1998), one of the first CNNs ever built! Key differences from modern models:
- **5x5 kernels** (bigger "eyes" looking at the image)
- **Average Pooling** (smoother downsampling instead of harsh max)
- Classic FC sizes: 120 -> 84 -> output

This is like using a vintage car in a modern race -- it still works, but how well?

~200K parameters (very small by modern standards)

In [ ]:
class LeNetCNN(nn.Module):
    """Model 8: LeNet-inspired CNN with 5x5 kernels and average pooling."""

    def __init__(self, input_channels: int = 3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            # Layer 1: big 5x5 filters (only 6 of them, like the original)
            nn.Conv2d(input_channels, 6, kernel_size=5),     # 150->146
            nn.ReLU(),
            nn.AvgPool2d(2),                                  # 146->73
            # Layer 2: more 5x5 filters
            nn.Conv2d(6, 16, kernel_size=5),                  # 73->69
            nn.ReLU(),
            nn.AvgPool2d(2),                                  # 69->34
        )
        # After 2 conv+pool with 5x5 kernels: (16, 34, 34)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 34 * 34, 120),    # Classic LeNet sizes
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, NUM_CLASSES),
        )

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        return self.classifier(x)

### Model 9: Transfer Learning with ResNet18

Instead of learning from scratch, we use **ResNet18** -- a model that already learned to "see" from **millions of ImageNet images**. We freeze all its layers and only train a small custom "head".

It is like hiring an expert photographer and only teaching them which bin to sort photos into (cat vs dog). This is called **transfer learning** and usually gives the best results with the least training.

~130K trainable parameters (out of ~11M total -- most are frozen)

**Note:** This model requires RGB (3-channel) input.

In [ ]:
class TransferResNet18(nn.Module):
    """Model 9: Pretrained ResNet18 with frozen backbone + custom head."""

    def __init__(self, input_channels: int = 3) -> None:
        if input_channels != 3:
            raise ValueError("TransferResNet18 requires RGB input (3 channels).")
        super().__init__()

        # Load ResNet18 pretrained on ImageNet (millions of images)
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Freeze all backbone layers -- do not update their weights
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Replace the final layer with our custom cat/dog head
        num_features = self.backbone.fc.in_features  # 512
        self.backbone.fc = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, NUM_CLASSES),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.backbone(x)

### Model 10: Lightweight CNN

The **smallest model** in our study. Uses **Global Average Pooling** instead of giant fully-connected layers -- this dramatically reduces the parameter count.

Global Average Pooling works like this: instead of flattening a 64x18x18 feature map into 20,736 numbers, it averages each channel down to a single number -- giving just 64 numbers. Much simpler!

~25-30K parameters (100x fewer than the baseline!)

In [ ]:
class LightweightCNN(nn.Module):
    """Model 10: Tiny CNN with Global Average Pooling -- fewest parameters."""

    def __init__(self, input_channels: int = 3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        # Global Average Pooling: reduces ANY spatial size to 1x1
        self.gap = nn.AdaptiveAvgPool2d(1)
        # Tiny classifier: just 64 -> 2
        self.classifier = nn.Linear(64, NUM_CLASSES)

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        x = self.gap(x)               # (batch, 64, 1, 1)
        x = x.view(x.size(0), -1)     # (batch, 64)
        return self.classifier(x)

### Model Registry & Summary Table

The `MODEL_REGISTRY` is a dictionary that maps model numbers to their class constructors. This lets us loop through all models easily during training. Below we also print a summary table showing each model's parameter count.

In [ ]:
# Registry: maps model number to (class, name, group)
MODEL_REGISTRY: dict = {
    1:  (BaselineCNN,      "Baseline CNN",      "A"),
    2:  (ShallowCNN,       "Shallow CNN",       "A"),
    3:  (DeepCNN,          "Deep CNN",          "A"),
    4:  (WideCNN,          "Wide CNN",          "A"),
    5:  (SmallFCCNN,       "Small FC CNN",      "C"),
    6:  (DropoutCNN,       "Dropout CNN",       "B"),
    7:  (BatchNormCNN,     "BatchNorm CNN",     "B"),
    8:  (LeNetCNN,         "LeNet-Style CNN",   "C"),
    9:  (TransferResNet18, "Transfer ResNet18", "D"),
    10: (LightweightCNN,   "Lightweight CNN",   "D"),
}


def count_parameters(model: nn.Module) -> int:
    """Count the total number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_all_parameters(model: nn.Module) -> int:
    """Count ALL parameters (trainable + frozen)."""
    return sum(p.numel() for p in model.parameters())


def format_params(n: int) -> str:
    """Format parameter count nicely: 3,450,000 -> '3.45M'."""
    if n >= 1_000_000:
        return f"{n / 1_000_000:.2f}M"
    if n >= 1_000:
        return f"{n / 1_000:.1f}K"
    return str(n)


def format_time(seconds: float) -> str:
    """Format seconds into a readable string like '2m 30s'."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes}m {secs:.0f}s"


# Print summary table of all 10 models
print("=" * 70)
print(f"{'#':<4} {'Model Name':<22} {'Group':<8} {'Trainable Params':<18} {'Total Params'}")
print("=" * 70)
for model_id, (cls, name, group) in MODEL_REGISTRY.items():
    try:
        model = cls(input_channels=3)
        trainable = count_parameters(model)
        total = count_all_parameters(model)
        print(f"{model_id:<4} {name:<22} {group:<8} {format_params(trainable):<18} {format_params(total)}")
        del model
    except Exception as e:
        print(f"{model_id:<4} {name:<22} {group:<8} Error: {e}")
print("=" * 70)

# Free GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Section 4: Training Infrastructure

### 4.1 Trainer Class

The Trainer handles the full process of teaching a model:
1. **Forward pass**: Show images to the model and get predictions
2. **Loss computation**: Check how wrong the model was
3. **Backward pass**: Calculate how to adjust the weights
4. **Update weights**: Make the model slightly less wrong

This repeats for many **epochs** (full passes through all the data). The Trainer also includes:
- **tqdm progress bars** so we can see training progress in real time
- **Early stopping** -- if the model stops improving for 5 epochs, we stop training early to save time
- **Learning rate scheduling** -- automatically reduces the learning step if the model plateaus

In [ ]:
class Trainer:
    """Trains a model and tracks metrics (accuracy, loss) per epoch."""

    def __init__(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        device: torch.device,
        lr: float = LR,
    ) -> None:
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device

        # Loss function: CrossEntropyLoss is standard for classification
        self.criterion = nn.CrossEntropyLoss()

        # Optimizer: Adam adjusts weights to minimize the loss
        # Only optimize parameters that require gradients (important for transfer learning)
        self.optimizer = Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr,
        )

        # Scheduler: reduce learning rate when the loss stops decreasing
        self.scheduler = ReduceLROnPlateau(
            self.optimizer, mode="min", patience=3, factor=0.5,
        )
        self.best_val_acc = 0.0

    def fit(self, num_epochs: int = EPOCHS) -> dict:
        """
        Train the model for num_epochs and return history.
        Returns a dict with lists: train_acc, val_acc, train_loss, val_loss,
        plus total_time and epochs_completed.
        """
        history: dict = {
            "train_acc": [], "val_acc": [],
            "train_loss": [], "val_loss": [],
        }
        patience_counter = 0
        best_val_loss = float("inf")
        start_time = time.time()

        for epoch in range(1, num_epochs + 1):
            # Train one epoch
            train_loss, train_acc = self._train_one_epoch(epoch, num_epochs)
            val_loss, val_acc = self._validate()

            # Record metrics
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            # Adjust learning rate based on validation loss
            self.scheduler.step(val_loss)

            # Track best accuracy
            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc

            # Early stopping: stop if val_loss hasn't improved
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1

            elapsed = time.time() - start_time
            print(f"  Epoch {epoch}/{num_epochs} - "
                  f"Train: {train_acc:.1%} | Val: {val_acc:.1%} | "
                  f"Loss: {val_loss:.4f} | Time: {format_time(elapsed)}")

            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"  Early stopping triggered (patience={EARLY_STOPPING_PATIENCE})")
                break

        total_time = time.time() - start_time
        history["total_time"] = total_time
        history["epochs_completed"] = len(history["train_acc"])
        print(f"  Training complete in {format_time(total_time)}")
        return history

    def _train_one_epoch(self, epoch: int, total: int) -> tuple:
        """Run one training epoch. Returns (avg_loss, accuracy)."""
        self.model.train()  # Enable training mode (dropout active, batchnorm updates)
        running_loss = 0.0
        correct = 0
        total_samples = 0

        pbar = tqdm(
            self.train_loader,
            desc=f"  Epoch {epoch}/{total} [Train]",
            leave=False,
        )
        for images, labels in pbar:
            images = images.to(self.device)
            labels = labels.to(self.device)

            # Forward pass: get predictions from the model
            self.optimizer.zero_grad()
            outputs = self.model(images)

            # Compute loss: how wrong was the model?
            loss = self.criterion(outputs, labels)

            # Backward pass: compute gradients
            loss.backward()

            # Update weights: make the model a bit less wrong
            self.optimizer.step()

            # Track statistics
            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total_samples += images.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = running_loss / total_samples
        accuracy = correct / total_samples
        return avg_loss, accuracy

    @torch.no_grad()  # Disable gradient computation for speed during validation
    def _validate(self) -> tuple:
        """Run validation pass. Returns (avg_loss, accuracy)."""
        self.model.eval()  # Disable training mode (dropout off)
        running_loss = 0.0
        correct = 0
        total_samples = 0

        for images, labels in self.val_loader:
            images = images.to(self.device)
            labels = labels.to(self.device)

            outputs = self.model(images)
            loss = self.criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total_samples += images.size(0)

        avg_loss = running_loss / total_samples
        accuracy = correct / total_samples
        return avg_loss, accuracy

### 4.2 Evaluator Class

After training, the Evaluator tests the model and calculates detailed metrics:
- **Accuracy**: What percentage of images were classified correctly
- **Precision**: Of all images predicted as "cat", how many really were cats
- **Recall**: Of all actual cats, how many did the model find
- **F1 Score**: A balanced combination of precision and recall
- **Confusion Matrix**: A 2x2 table showing correct vs wrong predictions

In [ ]:
class Evaluator:
    """Evaluates a trained model and computes detailed metrics."""

    @staticmethod
    @torch.no_grad()
    def get_predictions(
        model: nn.Module,
        dataloader: DataLoader,
        device: torch.device,
    ) -> tuple:
        """
        Run the model on all data and collect predictions.
        Returns (all_preds, all_labels, all_probs) as NumPy arrays.
        """
        model.eval()
        all_preds: list = []
        all_labels: list = []
        all_probs: list = []

        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
            all_probs.append(probs.cpu().numpy())

        return (
            np.concatenate(all_preds),
            np.concatenate(all_labels),
            np.concatenate(all_probs),
        )

    @staticmethod
    def evaluate_model(
        model: nn.Module,
        dataloader: DataLoader,
        device: torch.device,
    ) -> dict:
        """Full evaluation: accuracy, precision, recall, F1, confusion matrix."""
        preds, labels, probs = Evaluator.get_predictions(model, dataloader, device)

        cm = confusion_matrix(labels, preds)
        report = classification_report(
            labels, preds, target_names=CLASS_NAMES, output_dict=True,
        )
        accuracy = float(np.mean(preds == labels))

        return {
            "accuracy": accuracy,
            "precision_cat": report["cat"]["precision"],
            "recall_cat": report["cat"]["recall"],
            "f1_cat": report["cat"]["f1-score"],
            "precision_dog": report["dog"]["precision"],
            "recall_dog": report["dog"]["recall"],
            "f1_dog": report["dog"]["f1-score"],
            "confusion_matrix": cm,
            "predictions": preds,
            "labels": labels,
            "probabilities": probs,
        }

    @staticmethod
    def get_misclassified_samples(
        model: nn.Module,
        dataloader: DataLoader,
        device: torch.device,
        num: int = 8,
    ) -> tuple:
        """
        Find images the model got wrong.
        Returns (images, true_labels, pred_labels) for the first `num` mistakes.
        """
        model.eval()
        wrong_images: list = []
        wrong_true: list = []
        wrong_pred: list = []

        with torch.no_grad():
            for images, labels in dataloader:
                images_dev = images.to(device)
                outputs = model(images_dev)
                preds = outputs.argmax(dim=1).cpu()

                mask = preds != labels
                for idx in mask.nonzero(as_tuple=True)[0]:
                    i = idx.item()
                    wrong_images.append(images[i])
                    wrong_true.append(labels[i].item())
                    wrong_pred.append(preds[i].item())
                    if len(wrong_images) >= num:
                        return wrong_images, wrong_true, wrong_pred

        return wrong_images, wrong_true, wrong_pred

print("Trainer and Evaluator classes defined!")

## Section 5: Train All 10 Models

Now for the main event! We loop through all 10 models, train each one, and evaluate it. This is the most time-consuming part -- expect it to take 15-30 minutes on a Colab GPU.

Each model gets:
- 15 epochs of training (or fewer if early stopping kicks in)
- The same training and validation data
- The same learning rate and batch size

This makes the comparison fair.

In [ ]:
# Dictionary to store all results for every model
all_results: dict = {}

print("=" * 70)
print("TRAINING ALL 10 MODELS")
print("=" * 70)

for model_id, (model_cls, model_name, group) in MODEL_REGISTRY.items():
    print(f"\n{'='*70}")
    print(f"Model {model_id}/10: {model_name} (Group {group})")
    print(f"{'='*70}")

    # Reset seeds before each model for fair comparison
    set_seeds(SEED)

    try:
        # Create model instance
        model = model_cls(input_channels=3)
        trainable_params = count_parameters(model)
        total_params = count_all_parameters(model)
        print(f"  Trainable parameters: {format_params(trainable_params)}")
        print(f"  Total parameters:     {format_params(total_params)}")

        # Train the model
        trainer = Trainer(model, train_loader, val_loader, DEVICE, lr=LR)
        history = trainer.fit(num_epochs=EPOCHS)

        # Evaluate the trained model
        metrics = Evaluator.evaluate_model(model, val_loader, DEVICE)

        # Store everything in our results dictionary
        all_results[model_name] = {
            "model_id": model_id,
            "model": model,
            "group": group,
            "history": history,
            "metrics": metrics,
            "accuracy": metrics["accuracy"],
            "total_params": total_params,
            "trainable_params": trainable_params,
            "total_time": history["total_time"],
            "epochs_completed": history["epochs_completed"],
            "train_acc": history["train_acc"],
            "val_acc": history["val_acc"],
            "train_loss": history["train_loss"],
            "val_loss": history["val_loss"],
        }

        # Print a summary for this model
        print(f"\n  RESULT: {model_name}")
        print(f"  Accuracy:    {metrics['accuracy']:.1%}")
        print(f"  Precision:   Cat={metrics['precision_cat']:.3f}  Dog={metrics['precision_dog']:.3f}")
        print(f"  Recall:      Cat={metrics['recall_cat']:.3f}  Dog={metrics['recall_dog']:.3f}")
        print(f"  F1:          Cat={metrics['f1_cat']:.3f}  Dog={metrics['f1_dog']:.3f}")
        print(f"  Time:        {format_time(history['total_time'])}")
        print(f"  Epochs:      {history['epochs_completed']}/{EPOCHS}")

    except Exception as e:
        print(f"  ERROR training {model_name}: {e}")
        import traceback
        traceback.print_exc()

    # Free GPU memory between models
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*70}")
print(f"ALL TRAINING COMPLETE! Trained {len(all_results)} models successfully.")
print(f"{'='*70}")

## Section 6: Per-Model Results

Now let us visualize the results for each model individually. For each model we create:
1. **Training curves** -- accuracy and loss over epochs (to spot overfitting)
2. **Confusion matrix** -- a heatmap showing correct vs wrong predictions
3. **Predictions grid** -- actual images with green (correct) and red (wrong) labels

### 6.1 Visualization Helper Functions

These functions create the plots we need. They are defined once and reused for every model.

In [ ]:
def plot_training_summary(history: dict, model_name: str) -> None:
    """Create a 2-panel figure: accuracy curves + loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs_range = range(1, len(history["train_acc"]) + 1)

    # --- Accuracy panel ---
    ax1.plot(epochs_range, history["train_acc"], "b-o", label="Train", markersize=4)
    ax1.plot(epochs_range, history["val_acc"], "r-o", label="Validation", markersize=4)
    best_epoch = int(np.argmax(history["val_acc"])) + 1
    best_val = max(history["val_acc"])
    ax1.axvline(best_epoch, color="green", linestyle="--", alpha=0.5)
    ax1.annotate(
        f"Best: {best_val:.1%} (ep {best_epoch})",
        xy=(best_epoch, best_val), fontsize=9,
        xytext=(5, -15), textcoords="offset points",
    )
    ax1.set_title(f"Accuracy - {model_name}", fontweight="bold")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # --- Loss panel ---
    ax2.plot(epochs_range, history["train_loss"], "b-o", label="Train", markersize=4)
    ax2.plot(epochs_range, history["val_loss"], "r-o", label="Validation", markersize=4)
    ax2.set_title(f"Loss - {model_name}", fontweight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    safe_name = model_name.replace(" ", "_").lower()
    plt.savefig(GRAPHS_DIR / f"training_{safe_name}.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()


def plot_confusion_matrix_heatmap(cm: np.ndarray, model_name: str) -> None:
    """Draw a heatmap confusion matrix with counts and percentages."""
    total = cm.sum()
    # Build labels showing both the count and the percentage
    labels_arr = np.array([
        [f"{val}\n({val / total:.1%})" for val in row]
        for row in cm
    ])

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=labels_arr, fmt="",
        cmap="Blues", cbar=True,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        ax=ax, linewidths=0.5,
    )
    ax.set_title(f"Confusion Matrix - {model_name}", fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

    plt.tight_layout()
    safe_name = model_name.replace(" ", "_").lower()
    plt.savefig(GRAPHS_DIR / f"confusion_{safe_name}.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()


def plot_predictions_grid(
    images: list,
    true_labels: list,
    pred_labels: list,
    model_name: str = "",
    num_images: int = 12,
) -> None:
    """Grid of images: green title = correct, red title = wrong."""
    num_images = min(num_images, len(images))
    if num_images == 0:
        print(f"  No predictions to display for {model_name}")
        return
    cols = 4
    rows = max(1, (num_images + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
    axes_flat = axes.flatten()

    for i in range(num_images):
        img = tensor_to_image(images[i])
        axes_flat[i].imshow(img)
        true_name = CLASS_NAMES[true_labels[i]]
        pred_name = CLASS_NAMES[pred_labels[i]]
        correct = true_labels[i] == pred_labels[i]
        color = "green" if correct else "red"
        axes_flat[i].set_title(f"True: {true_name}\nPred: {pred_name}",
                               color=color, fontsize=10)
        axes_flat[i].axis("off")

    for i in range(num_images, len(axes_flat)):
        axes_flat[i].axis("off")

    title = f"Predictions - {model_name}" if model_name else "Predictions"
    plt.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    safe_name = model_name.replace(" ", "_").lower() or "model"
    plt.savefig(GRAPHS_DIR / f"predictions_{safe_name}.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()

print("Visualization functions defined!")

### 6.2 Generate All Per-Model Plots

Now we loop through every trained model and generate its individual result plots. For each model you will see:
- A 2-panel training summary (accuracy and loss curves)
- A confusion matrix heatmap
- A grid of sample predictions (green = correct, red = wrong)

In [ ]:
for model_name, result in all_results.items():
    print(f"\n{'='*60}")
    print(f"Results for: {model_name}")
    print(f"{'='*60}")

    # 1. Training curves (accuracy + loss over epochs)
    plot_training_summary(result["history"], model_name)

    # 2. Confusion matrix heatmap
    plot_confusion_matrix_heatmap(result["metrics"]["confusion_matrix"], model_name)

    # 3. Predictions grid -- collect some sample images to display
    model = result["model"]
    sample_images = []
    sample_true = []
    sample_pred = []
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            images_dev = images.to(DEVICE)
            outputs = model(images_dev)
            batch_preds = outputs.argmax(dim=1).cpu()

            for i in range(len(images)):
                if len(sample_images) < 12:
                    sample_images.append(images[i])
                    sample_true.append(labels[i].item())
                    sample_pred.append(batch_preds[i].item())
            if len(sample_images) >= 12:
                break

    plot_predictions_grid(sample_images, sample_true, sample_pred, model_name, num_images=12)

print("\nAll per-model plots generated!")

## Section 7: Cross-Model Comparison

This is where it gets really interesting! We compare all 10 models side-by-side to answer the big questions:
- Which model is most accurate?
- Does bigger always mean better?
- How do the research groups compare?

### 7.1 Accuracy Comparison (Sorted Best to Worst)

The most important chart -- which model correctly identifies the most cats and dogs?

In [ ]:
def plot_accuracy_comparison(all_results: dict) -> None:
    """Bar chart comparing validation accuracy, sorted best to worst."""
    names = list(all_results.keys())
    values = [all_results[n]["accuracy"] * 100 for n in names]

    # Sort by value descending (best first)
    order = np.argsort(values)[::-1]
    names_sorted = [names[i] for i in order]
    values_sorted = [values[i] for i in order]
    colors_sorted = [MODEL_COLORS[list(all_results.keys()).index(n) % 10] for n in names_sorted]

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.bar(names_sorted, values_sorted, color=colors_sorted)

    # Add value labels on top of each bar
    for bar, val in zip(bars, values_sorted):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_title("Validation Accuracy Comparison (Sorted Best to Worst)", fontweight="bold", fontsize=14)
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 105)
    plt.xticks(rotation=35, ha="right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / "comparison_accuracy.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()

plot_accuracy_comparison(all_results)

### 7.2 Loss Comparison

Lower loss means the model is more confident in its correct predictions. A model with high accuracy but also high loss might be "lucky" rather than truly confident.

In [ ]:
def plot_loss_comparison(all_results: dict) -> None:
    """Bar chart comparing final validation loss of all models."""
    names = list(all_results.keys())
    values = [r["val_loss"][-1] for r in all_results.values()]

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.bar(names, values, color=MODEL_COLORS[:len(names)])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{val:.4f}", ha="center", va="bottom", fontsize=8)

    ax.set_title("Final Validation Loss Comparison", fontweight="bold", fontsize=14)
    ax.set_ylabel("Loss")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / "comparison_loss.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()

plot_loss_comparison(all_results)

### 7.3 Parameter Count Comparison

How many "knobs" does each model have? More parameters means more capacity to learn, but also higher risk of overfitting and slower training.

In [ ]:
def plot_params_comparison(all_results: dict) -> None:
    """Bar chart comparing parameter counts across all models."""
    names = list(all_results.keys())
    values = [r["total_params"] for r in all_results.values()]

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.bar(names, values, color=MODEL_COLORS[:len(names)])
    for bar, val in zip(bars, values):
        label = f"{val / 1e6:.2f}M" if val >= 1e6 else f"{val / 1e3:.1f}K"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                label, ha="center", va="bottom", fontsize=8)

    ax.set_title("Parameter Count Comparison", fontweight="bold", fontsize=14)
    ax.set_ylabel("Total Parameters")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / "comparison_params.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()

plot_params_comparison(all_results)

### 7.4 Training Time Comparison

Time is money! Which models train the fastest? Faster training means quicker experimentation.

In [ ]:
def plot_time_comparison(all_results: dict) -> None:
    """Bar chart comparing training times across all models."""
    names = list(all_results.keys())
    values = [r["total_time"] for r in all_results.values()]

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.bar(names, values, color=MODEL_COLORS[:len(names)])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{val:.0f}s", ha="center", va="bottom", fontsize=8)

    ax.set_title("Training Time Comparison", fontweight="bold", fontsize=14)
    ax.set_ylabel("Time (seconds)")
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / "comparison_time.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()

plot_time_comparison(all_results)

### 7.5 Accuracy vs Parameters Scatter Plot

This is the most insightful chart: it shows whether bigger models (more parameters) are actually more accurate. Points are color-coded by research group.

The ideal model would be in the **top-left corner** -- high accuracy with few parameters.

In [ ]:
def plot_accuracy_vs_params(all_results: dict) -> None:
    """Scatter plot: accuracy vs parameter count, color-coded by group."""
    fig, ax = plt.subplots(figsize=(10, 7))

    for name, r in all_results.items():
        group = r.get("group", "A")
        color = GROUP_COLORS.get(group, "#999999")
        ax.scatter(
            r["total_params"], r["accuracy"] * 100,
            color=color, s=150, zorder=5, edgecolors="black", linewidth=0.5,
        )
        ax.annotate(name, (r["total_params"], r["accuracy"] * 100),
                    fontsize=8, ha="left", va="bottom",
                    xytext=(5, 5), textcoords="offset points")

    ax.set_title("Accuracy vs Parameter Count", fontweight="bold", fontsize=14)
    ax.set_xlabel("Total Parameters")
    ax.set_ylabel("Validation Accuracy (%)")
    ax.grid(True, alpha=0.3)

    # Legend for research groups
    group_labels = {
        "A": "Group A (Depth Study)",
        "B": "Group B (Regularization)",
        "C": "Group C (Architecture)",
        "D": "Group D (Advanced)",
    }
    for group, color in GROUP_COLORS.items():
        ax.scatter([], [], color=color, label=group_labels[group], s=100,
                   edgecolors="black", linewidth=0.5)
    ax.legend(loc="lower right")

    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / "comparison_acc_vs_params.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()

plot_accuracy_vs_params(all_results)

### 7.6 Group Average Accuracy

How do the research groups compare overall? This averages the accuracy of all models within each group to see which approach works best on average.

In [ ]:
def plot_group_comparison(all_results: dict) -> None:
    """Bar chart showing average accuracy per research group."""
    groups: dict = {}
    for r in all_results.values():
        g = r.get("group", "A")
        groups.setdefault(g, []).append(r["accuracy"] * 100)

    group_names = sorted(groups.keys())
    avgs = [np.mean(groups[g]) for g in group_names]
    colors = [GROUP_COLORS.get(g, "#999") for g in group_names]

    group_labels = {
        "A": "Group A\n(Depth Study)",
        "B": "Group B\n(Regularization)",
        "C": "Group C\n(Architecture)",
        "D": "Group D\n(Advanced)",
    }

    fig, ax = plt.subplots(figsize=(8, 5))
    x_labels = [group_labels.get(g, f"Group {g}") for g in group_names]
    bars = ax.bar(x_labels, avgs, color=colors)
    for bar, val in zip(bars, avgs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

    ax.set_title("Average Accuracy by Research Group", fontweight="bold", fontsize=14)
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / "comparison_groups.png", dpi=GRAPH_DPI, bbox_inches='tight')
    plt.show()

plot_group_comparison(all_results)

## Section 8: Results Collection for README

This section prints all results in a clean, copy-paste-ready markdown format for the project README file. Just copy everything between the markers.

In [ ]:
print("\n")
print("=" * 80)
print("COPY EVERYTHING BELOW FOR README")
print("=" * 80)
print()

# --- Dataset Overview ---
print("## Dataset Overview")
print()
print(f"- **Dataset**: Cats and Dogs Filtered (Google ML Education)")
print(f"- **Training images**: {len(train_dataset)}")
print(f"- **Validation images**: {len(val_dataset)}")
print(f"- **Image size**: {IMAGE_SIZE}x{IMAGE_SIZE} pixels (RGB)")
print(f"- **Classes**: {', '.join(CLASS_NAMES)}")
print()

# --- Per-Model Results Table ---
print("## Model Results")
print()
print("| # | Model | Accuracy | Params | Time | Epochs | Group |")
print("|---|-------|----------|--------|------|--------|-------|")
for name, r in all_results.items():
    mid = r["model_id"]
    acc = r["accuracy"]
    params = r["total_params"]
    t = r["total_time"]
    ep = r["epochs_completed"]
    grp = r["group"]
    print(f"| {mid} | {name} | {acc:.1%} | {format_params(params)} | {format_time(t)} | {ep}/{EPOCHS} | {grp} |")
print()

# --- Sorted by accuracy ---
print("## Rankings (by Validation Accuracy)")
print()
sorted_models = sorted(all_results.items(), key=lambda x: x[1]["accuracy"], reverse=True)

print("| Rank | Model | Accuracy |")
print("|------|-------|----------|")
for rank, (name, r) in enumerate(sorted_models, 1):
    print(f"| {rank} | {name} | {r['accuracy']:.1%} |")
print()

# --- Top 3 and Bottom 3 ---
print("### Top 3 Models")
print()
for i, (name, r) in enumerate(sorted_models[:3], 1):
    print(f"{i}. **{name}** - {r['accuracy']:.1%} accuracy, {format_params(r['total_params'])} params")
print()

print("### Bottom 3 Models")
print()
for i, (name, r) in enumerate(sorted_models[-3:], len(sorted_models) - 2):
    print(f"{i}. **{name}** - {r['accuracy']:.1%} accuracy, {format_params(r['total_params'])} params")
print()

# --- Group Analysis ---
print("## Group Analysis")
print()
groups_data: dict = {}
for name, r in all_results.items():
    g = r["group"]
    groups_data.setdefault(g, []).append((name, r["accuracy"]))

group_descriptions = {
    "A": "Depth Study (Baseline, Shallow, Deep, Wide)",
    "B": "Regularization (Dropout, BatchNorm)",
    "C": "Architecture (Small FC, LeNet-Style)",
    "D": "Advanced (Transfer ResNet18, Lightweight)",
}

print("| Group | Description | Avg Accuracy | Models |")
print("|-------|-------------|-------------|--------|")
for g in sorted(groups_data.keys()):
    desc = group_descriptions.get(g, "Unknown")
    avg = np.mean([acc for _, acc in groups_data[g]]) * 100
    model_list = ", ".join([name for name, _ in groups_data[g]])
    print(f"| {g} | {desc} | {avg:.1f}% | {model_list} |")
print()

# --- Detailed per-model metrics ---
print("## Detailed Metrics")
print()
print("| Model | Precision (Cat) | Recall (Cat) | F1 (Cat) | Precision (Dog) | Recall (Dog) | F1 (Dog) |")
print("|-------|----------------|-------------|---------|----------------|-------------|---------|")
for name, r in all_results.items():
    m = r["metrics"]
    print(f"| {name} | {m['precision_cat']:.3f} | {m['recall_cat']:.3f} | {m['f1_cat']:.3f} | "
          f"{m['precision_dog']:.3f} | {m['recall_dog']:.3f} | {m['f1_dog']:.3f} |")
print()

# --- Key Findings ---
best_name, best_r = sorted_models[0]
worst_name, worst_r = sorted_models[-1]
smallest = min(all_results.items(), key=lambda x: x[1]["total_params"])
largest = max(all_results.items(), key=lambda x: x[1]["total_params"])
fastest = min(all_results.items(), key=lambda x: x[1]["total_time"])
slowest = max(all_results.items(), key=lambda x: x[1]["total_time"])

print("## Key Findings")
print()
print(f"1. **Best model**: {best_name} with {best_r['accuracy']:.1%} accuracy")
print(f"2. **Worst model**: {worst_name} with {worst_r['accuracy']:.1%} accuracy")
print(f"3. **Smallest model**: {smallest[0]} with {format_params(smallest[1]['total_params'])} parameters")
print(f"4. **Largest model**: {largest[0]} with {format_params(largest[1]['total_params'])} parameters")
print(f"5. **Fastest training**: {fastest[0]} in {format_time(fastest[1]['total_time'])}")
print(f"6. **Slowest training**: {slowest[0]} in {format_time(slowest[1]['total_time'])}")
print(f"7. **Accuracy range**: {worst_r['accuracy']:.1%} to {best_r['accuracy']:.1%}")
print(f"8. **Parameter range**: {format_params(smallest[1]['total_params'])} to {format_params(largest[1]['total_params'])}")
print()

# --- Training Configuration ---
print("## Training Configuration")
print()
print(f"- **Epochs**: {EPOCHS} (with early stopping, patience={EARLY_STOPPING_PATIENCE})")
print(f"- **Batch size**: {BATCH_SIZE}")
print(f"- **Learning rate**: {LR}")
print(f"- **Optimizer**: Adam")
print(f"- **Loss function**: CrossEntropyLoss")
print(f"- **LR Scheduler**: ReduceLROnPlateau (patience=3, factor=0.5)")
print(f"- **Device**: {DEVICE}")
print(f"- **Seed**: {SEED}")
print()
print("=" * 80)
print("END OF README CONTENT")
print("=" * 80)

## Done!

All 10 models have been trained, evaluated, and compared. The results and graphs have been saved to the `results/graphs/` directory.

**Summary of what we learned:**
- Transfer learning (ResNet18) typically gives the best results with the least training effort
- Regularization techniques (Dropout, BatchNorm) help prevent overfitting
- More parameters do not always mean better accuracy
- The Lightweight CNN proves that smart architecture design can beat brute force